# F. pandas for Biomedical Tables

I use NumPy for waveform calculations and pandas for beat annotations, metadata, and feature tables. A `Series` is one labeled column; a `DataFrame` combines columns into a table. The small teaching table below is not patient data.

## F.1 Build a beat table

Each row represents one beat. I put units in column names to distinguish sample positions from time values.

In [1]:
import numpy as np
import pandas as pd
fs = 360
beats = pd.DataFrame({"sample": [180, 468, 720, 1035], "label": ["N", "N", "V", "N"]})
beats["time_s"] = beats["sample"] / fs
beats["rr_s"] = beats["time_s"].diff()
print(beats.to_string(index=False))

 sample label  time_s  rr_s
    180     N   0.500   NaN
    468     N   1.300 0.800
    720     V   2.000 0.700
   1035     N   2.875 0.875


The first RR interval is missing because no earlier beat appears in this table. Each later interval ends at the beat on that row. If a detection is missing, an interval can span multiple heartbeats; the table does not verify detection quality.

## F.2 Select rows and columns

`.loc` selects by labels or a Boolean condition. `.iloc` selects by integer position. Use explicit column names so the result is easy to inspect.

In [2]:
normal = beats.loc[beats["label"] == "N", ["sample", "time_s", "rr_s"]]
print("Normal-labeled rows:")
print(normal.to_string(index=False))
print("First two rows by position:")
print(beats.iloc[:2].to_string(index=False))

Normal-labeled rows:
 sample  time_s  rr_s
    180   0.500   NaN
    468   1.300 0.800
   1035   2.875 0.875
First two rows by position:
 sample label  time_s  rr_s
    180     N     0.5   NaN
    468     N     1.3   0.8


An `N` label on the current row does not guarantee a normal-to-normal interval. The previous beat must also be normal.

In [3]:
normal_pair = beats["label"].eq("N") & beats["label"].shift().eq("N")
print("NN intervals (s):", beats.loc[normal_pair, "rr_s"].to_list())
print("Counts by beat label:")
print(beats.groupby("label").size().to_string())

NN intervals (s): [0.8]
Counts by beat label:
label
N    3
V    1


## F.3 Missing data and numerical arrays

Use `isna()` to inspect missing values. `dropna()` can select complete values for a particular summary; it does not reconstruct missing physiology. `to_numpy()` supplies numerical values without the row labels.

In [4]:
print("Missing values:")
print(beats.isna().sum().to_string())
rr = beats["rr_s"].dropna().to_numpy(dtype=float)
print("Available intervals:", rr)
print("Mean interval (s):", np.round(rr.mean(), 4))

Missing values:
sample    0
label     0
time_s    0
rr_s      1
Available intervals: [0.8   0.7   0.875]
Mean interval (s): 0.7917


## F.4 CSV files and table joins

CSV stores text, so check types after reading it. Record identifiers often belong in string columns to preserve leading zeros. This in-memory example avoids creating a file.

In [5]:
from io import StringIO
csv_text = beats.to_csv(index=False)
loaded = pd.read_csv(StringIO(csv_text), dtype={"sample": "int64", "label": "string"})
print(loaded.to_string(index=False))
print("Rows preserved:", len(loaded) == len(beats))

 sample label  time_s  rr_s
    180     N   0.500   NaN
    468     N   1.300 0.800
    720     V   2.000 0.700
   1035     N   2.875 0.875
Rows preserved: True


When joining feature tables, use explicit keys and check their expected uniqueness. Duplicate keys can multiply rows. Preserve patient and record identifiers so that later model evaluation can separate patients rather than randomly mixing their beats.

## Further reading

- [pandas introductory guide](https://pandas.pydata.org/docs/user_guide/10min.html)
- [Missing data in pandas](https://pandas.pydata.org/docs/user_guide/missing_data.html)